In [10]:
import os
import pandas as pd
from pathlib import Path
import gc

# Run garbage collection
gc.collect()

data_folder = Path("/kaggle/input/competitions/name-that-llm/train.csv")
data_path = os.path.join(os.getcwd(), data_folder)

print(data_path)

# # submission data format
# df = pd.read_parquet(data_path)
# df.head()

df = pd.read_csv(data_path)
df.head()

/kaggle/input/competitions/name-that-llm/train.csv


,id,prompt_id,text,model
0,0018b3765972,p081,"When a language dies, a particular astronomy d...",qwen3.8-max
1,0041ac86ee93,p073,Data science communities matter because they a...,claude-sonnet-5
2,005a19083eae,p120,"In the final twilight of time, the last embers...",gemini-3.6-flash
3,009319596f10,p073,Data science communities matter because they t...,gpt-5.6-terra
4,00d3e8c92596,p097,"Deep inside each ear, three tiny loops of bone...",glm-5.3-flash


In [ ]:
!pip install -q transformers datasets accelerate evaluate


In [ ]:
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from transformers import DataCollatorWithPadding
import evaluate

# 1. Load Data
df = pd.read_csv("data/train.csv") # Kaggle path: /kaggle/input/competitions/name-that-llm/train.csv

# 2. Encode Labels (0 to 17)
le = LabelEncoder()
df["label"] = le.fit_transform(df["model"])
num_classes = len(le.classes_)

# 3. Stratified Train-Val Split (important taaki har model ka equal balance ho)
df_train, df_val = train_test_split(
    df, 
    test_size=0.2, 
    random_state=42, 
    stratify=df["label"]
)

# 4. Tokenizer Setup
MODEL_NAME = "microsoft/deberta-v3-small"  # GPU memory kam ho toh 'small', acchi GPU ho toh 'base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


In [ ]:
class LLMDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            padding=False, # DataCollator dynamically pad karega for speed
            return_tensors="pt"
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = LLMDataset(df_train["text"].values, df_train["label"].values, tokenizer)
val_dataset = LLMDataset(df_val["text"].values, df_val["label"].values, tokenizer)


In [ ]:
# Metrics: Accuracy & Macro Recall
accuracy_metric = evaluate.load("accuracy")
recall_metric = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    rec = recall_metric.compute(predictions=preds, references=labels, average="macro")["recall"]
    return {"accuracy": acc, "recall": rec}

# Model Load
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=num_classes
)

# Training Hyperparameters tuned for DeBERTa
training_args = TrainingArguments(
    output_dir="./deberta_llm_output",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,              # Transformers ke liye small LR zaroori hai (2e-5 se 3e-5)
    per_device_train_batch_size=8,   # GPU memory ke according 8 ya 16
    per_device_eval_batch_size=16,
    num_train_epochs=4,              # 3-5 epochs kafi hoti hain
    weight_decay=0.01,               # Overfitting rokne ke liye
    warmup_ratio=0.1,                # Learning rate warmup
    fp16=torch.cuda.is_available(),  # Fast GPU training
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

# Start Fine-tuning
trainer.train()


In [ ]:
# Validation dataset par evaluation run karein
eval_results = trainer.evaluate()

print(f"Final Accuracy: {eval_results['eval_accuracy']:.4f}")
print(f"Final Recall:   {eval_results['eval_recall']:.4f}")
print(f"Validation Loss: {eval_results['eval_loss']:.4f}")


Accuracy Score: 0.0204
Recall Score: 0.0208


In [ ]:
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix, classification_report
import numpy as np

# 1. Validation dataset par predictions nikalein
predictions_output = trainer.predict(val_dataset)

# 2. Raw logits ko class indices (0 to 17) mein convert karein
# predictions_output.predictions shape: (N, 18)
y_pred = np.argmax(predictions_output.predictions, axis=1)
y_true = predictions_output.label_ids

# 3. Calculate Scores
acc = accuracy_score(y_true, y_pred)
rec = recall_score(y_true, y_pred, average="macro")

print(f"Validation Accuracy: {acc:.4f}")
print(f"Validation Recall (Macro): {rec:.4f}")

# 4. Detailed Classification Report (Per-model breakdown)
print("\n--- Detailed Per-Class Report ---")
print(classification_report(y_true, y_pred, target_names=le.classes_))


In [ ]:
# 1. Test dataset prepare karein
test_df = pd.read_csv(r"C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\data\test.csv") # ya Kaggle path
test_dataset = LLMDataset(test_df["text"].values, labels=None, tokenizer=tokenizer)

# 2. Predict logits
test_preds_output = trainer.predict(test_dataset)
test_pred_indices = np.argmax(test_preds_output.predictions, axis=1)

# 3. Inverse transform (numbers se wapas actual model names mein convert karein)
test_predictions = le.inverse_transform(test_pred_indices)

# 4. Save to submission format
test_df["model"] = test_predictions
# test_df[["id", "model"]].to_csv("submission.csv", index=False)
test_df.head()
